In [ ]:
pip install fastapi uvicorn httpx

In [3]:
import os, sys, time, json, asyncio, statistics, subprocess
import httpx

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DTYPE = "float32"            # whichever was faster for you
MAX_NEW = 48
N_REQUESTS = 16              # requests per test
CONCURRENCY_LEVELS = [1, 4, 8]

QUESTIONS = [
    "Answer with one word: what colour is the sky on a clear day?",
    "What is 12 times 7? Reply with only the number.",
    "Write three sentences about why the ocean is salty.",
    "List five fruits, comma separated.",
    "Explain in detail how a refrigerator keeps food cold.",
    "Say hello.",
    "Give a short definition of photosynthesis.",
    "Write a short paragraph about the history of the bicycle.",
    "What is the capital of Japan? One word.",
    "Describe how rain forms, step by step.",
    "Translate 'good morning' into French.",
    "Write a haiku about coffee.",
    "Explain what a database index is and why it speeds up queries.",
    "Name the planet closest to the sun.",
    "Give three tips for writing clean Python code.",
    "Summarise the water cycle in two sentences.",
]
print("ready")

ready


In [4]:
def start_server(mode, port, max_batch=8, max_wait_ms=30):
    env = dict(os.environ, MODEL_ID=MODEL_ID, MODE=mode, DTYPE=DTYPE,
               MAX_BATCH=str(max_batch), MAX_WAIT_MS=str(max_wait_ms))
    log_path = f"server_{mode}_{port}.log"
    log = open(log_path, "w")
    proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "server_hf:app",
                             "--host", "127.0.0.1", "--port", str(port), "--log-level", "warning"],
                            env=env, stdout=log, stderr=subprocess.STDOUT)
    url = f"http://127.0.0.1:{port}"
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < 600:                    # model loading can take a while
        if proc.poll() is not None:
            log.close()
            raise RuntimeError("server crashed:\n" + open(log_path).read()[-3000:])
        try:
            if httpx.get(url + "/health", timeout=2).status_code == 200:
                print(f"{mode} server up on {url} after {time.perf_counter() - t0:.1f}s:",
                      httpx.get(url + "/health").json())
                return proc, url
        except httpx.HTTPError:
            pass
        time.sleep(1)
    proc.terminate()
    raise TimeoutError("server did not start; see " + log_path)


def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()
    print("server stopped")

In [5]:
async def one_request(client, url, question):
    t0 = time.perf_counter()
    r = await client.post(url + "/generate", json={"prompt": question, "max_new_tokens": MAX_NEW})
    r.raise_for_status()
    d = r.json()
    d["latency_s"] = time.perf_counter() - t0          # what the USER experiences (end to end)
    return d

async def load_test(url, concurrency, n_requests=N_REQUESTS):
    """Closed loop: `concurrency` virtual users, each sends its next request as soon as the last one returns."""
    todo = list(range(n_requests))
    results = []

    async def user(client):
        while todo:
            i = todo.pop(0)
            results.append(await one_request(client, url, QUESTIONS[i % len(QUESTIONS)]))

    async with httpx.AsyncClient(timeout=600) as client:
        await one_request(client, url, "Say hi.")      # warm-up, not counted
        t0 = time.perf_counter()
        await asyncio.gather(*[user(client) for _ in range(concurrency)])
        wall = time.perf_counter() - t0

    lat = sorted(r["latency_s"] for r in results)
    p = lambda q: lat[min(len(lat) - 1, int(q * len(lat)))]
    tokens = sum(r["new_tokens"] for r in results)
    return {"concurrency": concurrency, "requests": len(results), "wall_s": wall,
            "tok_per_s": tokens / wall, "req_per_s": len(results) / wall,
            "p50_s": p(0.50), "p95_s": p(0.95),
            "mean_queue_wait_s": statistics.mean(r["queue_wait_s"] for r in results),
            "mean_gen_s": statistics.mean(r["gen_s"] for r in results),
            "mean_batch_size": statistics.mean(r["batch_size"] for r in results)}

def print_row(mode, r):
    print(f"{mode:6s} | conc {r['concurrency']:>2} | {r['tok_per_s']:7.1f} tok/s | {r['req_per_s']:5.2f} req/s | "
          f"p50 {r['p50_s']:6.2f}s | p95 {r['p95_s']:6.2f}s | queue wait {r['mean_queue_wait_s']:6.2f}s | "
          f"gen {r['mean_gen_s']:5.2f}s | batch {r['mean_batch_size']:4.1f}")


In [6]:
all_results = {}
proc, url = start_server("single", 8001)
try:
    all_results["single"] = []
    for c in CONCURRENCY_LEVELS:
        r = await load_test(url, c)
        all_results["single"].append(r)
        print_row("single", r)
finally:
    stop_server(proc)

single server up on http://127.0.0.1:8001 after 20.2s: {'status': 'ok', 'mode': 'single', 'max_batch': 8, 'max_wait_ms': 30.0}
single | conc  1 |    12.7 tok/s |  0.45 req/s | p50   2.93s | p95   4.23s | queue wait   0.00s | gen  2.23s | batch  1.0
single | conc  4 |    12.8 tok/s |  0.45 req/s | p50   8.70s | p95  11.79s | queue wait   5.45s | gen  2.21s | batch  1.0
single | conc  8 |    12.8 tok/s |  0.45 req/s | p50  17.23s | p95  20.48s | queue wait  10.73s | gen  2.22s | batch  1.0
server stopped


In [7]:
proc, url = start_server("batch", 8002, max_batch=8, max_wait_ms=30)
try:
    all_results["batch"] = []
    for c in CONCURRENCY_LEVELS:
        r = await load_test(url, c)
        all_results["batch"].append(r)
        print_row("batch", r)
    print("batch sizes the server actually formed:", httpx.get(url + "/stats").json()["batch_sizes"])
finally:
    stop_server(proc)

batch server up on http://127.0.0.1:8002 after 11.5s: {'status': 'ok', 'mode': 'batch', 'max_batch': 8, 'max_wait_ms': 30.0}
batch  | conc  1 |    13.6 tok/s |  0.48 req/s | p50   2.87s | p95   3.97s | queue wait   0.03s | gen  2.05s | batch  1.0
batch  | conc  4 |    19.6 tok/s |  0.69 req/s | p50   5.77s | p95   5.87s | queue wait   0.03s | gen  5.74s | batch  4.0
batch  | conc  8 |    27.0 tok/s |  0.95 req/s | p50   8.72s | p95   8.72s | queue wait   0.00s | gen  8.38s | batch  8.0
batch sizes the server actually formed: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 1, 8, 8]
server stopped


In [8]:
print(f"{'conc':>4} | {'single tok/s':>12} | {'batch tok/s':>11} | {'single p95':>10} | {'batch p95':>9}")
for s, b in zip(all_results["single"], all_results["batch"]):
    print(f"{s['concurrency']:>4} | {s['tok_per_s']:>12.1f} | {b['tok_per_s']:>11.1f} | "
          f"{s['p95_s']:>9.2f}s | {b['p95_s']:>8.2f}s")
with open("results_04_api.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("saved results_04_api.json")

conc | single tok/s | batch tok/s | single p95 | batch p95
   1 |         12.7 |        13.6 |      4.23s |     3.97s
   4 |         12.8 |        19.6 |     11.79s |     5.87s
   8 |         12.8 |        27.0 |     20.48s |     8.72s
saved results_04_api.json


In [9]:
proc, url = start_server("batch", 8003, max_batch=8, max_wait_ms=0)
try:
    r = await load_test(url, 8)
    print_row("wait=0", r)
    print("batch sizes formed:", httpx.get(url + "/stats").json()["batch_sizes"])
finally:
    stop_server(proc)

proc, url = start_server("batch", 8004, max_batch=8, max_wait_ms=300)
try:
    r = await load_test(url, 8)
    print_row("wait=300", r)
    print("batch sizes formed:", httpx.get(url + "/stats").json()["batch_sizes"])
finally:
    stop_server(proc)

batch server up on http://127.0.0.1:8003 after 10.5s: {'status': 'ok', 'mode': 'batch', 'max_batch': 8, 'max_wait_ms': 0.0}
wait=0 | conc  8 |    28.0 tok/s |  0.99 req/s | p50   7.95s | p95   8.37s | queue wait   0.80s | gen  6.77s | batch  6.2
batch sizes formed: [1, 1, 7, 1, 7]
server stopped
batch server up on http://127.0.0.1:8004 after 10.3s: {'status': 'ok', 'mode': 'batch', 'max_batch': 8, 'max_wait_ms': 300.0}
wait=300 | conc  8 |    27.6 tok/s |  0.98 req/s | p50   8.21s | p95   8.21s | queue wait   0.00s | gen  8.17s | batch  8.0
batch sizes formed: [1, 8, 8]
server stopped
